In [18]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams["font.family"] = "Hiragino Sans"

connection = sqlite3.connect("notices.sqlite")

In [20]:
service_notices = pd.read_sql_query("""
       SELECT
           key,
           project_name,
           organization_name,
           prefecture_name,
           category,
           cft_issue_date,
           external_document_uri,
           data_json
       FROM notices
       WHERE category = '役務'
   """, connection)

service_notices.shape

(6000, 8)

In [22]:
service_notices[
       [
           "project_name",
           "organization_name",
           "prefecture_name",
           "cft_issue_date",
       ]
   ].head(20)

,project_name,organization_name,prefecture_name,cft_issue_date
0,一般競争入札の実施（令和８年度 農道設計指針資料作成委託業務）,北海道,北海道,2026-07-24T00:00:00+09:00
1,児童館運営支援システム導入及びシステム運用保守業務委託契約プロポーザルの実施について,東京都世田谷区,東京都,2026-07-24T00:00:00+09:00
2,宇宙戦略基金技術開発課題「テラヘルツ波リモートセンシング衛星による⽉地下浅部の資源探査」技術...,国立大学法人東京工業大学,東京都,2026-07-24T00:00:00+09:00
3,ヒト遺伝学的検査業務委託,国立研究開発法人国立成育医療研究センター,東京都,2026-07-24T00:00:00+09:00
4,【入札公告】給食・食器洗浄業務委託,独立行政法人国立病院機構村山医療センター,東京都,2026-07-24T00:00:00+09:00
5,令和８年度CE×CNの同時達成に向けた木材再利用の方策等検証事業委託業務【総合評価落札方式】,環境省,東京都,2026-07-24T00:00:00+09:00
6,AIテストベッド機能追加,国立研究開発法人産業技術総合研究所,東京都,2026-07-24T00:00:00+09:00
7,空調機器等整備役務,防衛省,東京都,2026-07-22T00:00:00+09:00
8,【税込入札】国際平和と安全シンポジウム（ハイブリッド形式）の会議運営等業務委託,防衛省,東京都,2026-07-21T00:00:00+09:00
9,令和８年度自動車事故による重度後遺障害者に係る支援策等の検討に関する調査 【業務委託】,国土交通省,東京都,2026-07-21T00:00:00+09:00


In [35]:
### Subgroups IT related services could belong to 


it_patterns = {
       "system_development": (
           r"(?:システム|情報システム).*"
           r"(?:開発|構築|改修|更新|導入)"
           r"|(?:開発|構築|改修|更新|導入).*"
           r"(?:システム|情報システム)"
       ),
       "software_applications": (
           r"ソフトウェア|アプリケーション|"
           r"ウェブアプリ|Webアプリ"
       ),
       "operations_maintenance": (
           r"(?:システム|情報システム).*"
           r"(?:運用|保守)"
           r"|(?:運用|保守).*"
           r"(?:システム|情報システム)"
       ),
       "cloud": (
           r"クラウド|SaaS|IaaS|PaaS"
       ),
       "cybersecurity": (
           r"サイバーセキュリティ|"
           r"情報セキュリティ|"
           r"セキュリティ対策"
       ),
       "network_infrastructure": (
           r"ネットワーク|サーバー?|データセンター"
       ),
       "data_digital": (
           r"データ(?:分析|基盤|連携|移行)|"
           r"デジタル化|DX"
       ),
       "artificial_intelligence": (
           r"人工知能|生成AI|生成ＡＩ|"
           r"(?<![A-Za-z])AI(?![A-Za-z])|ＡＩ|"
           r"機械学習|深層学習|"
           r"大規模言語モデル|"
           r"(?<![A-Za-z])LLM(?![A-Za-z])|"
           r"ChatGPT"
       ),
   }

In [36]:
for subgroup, pattern in it_patterns.items():
       service_notices[subgroup] = (
           service_notices["project_name"]
           .str.contains(
               pattern,
               case=False,
               na=False,
               regex=True,
           )
       )

tag_columns = list(it_patterns)

service_notices["it_related"] = (
       service_notices[tag_columns]
       .any(axis=1)
   )

In [37]:
service_notices[
       [
           "project_name",
           *tag_columns,
           "it_related",
       ]
].head(20)

,project_name,system_development,software_applications,operations_maintenance,cloud,cybersecurity,network_infrastructure,data_digital,artificial_intelligence,it_related
0,一般競争入札の実施（令和８年度 農道設計指針資料作成委託業務）,False,False,False,False,False,False,False,False,False
1,児童館運営支援システム導入及びシステム運用保守業務委託契約プロポーザルの実施について,True,False,True,False,False,False,False,False,True
2,宇宙戦略基金技術開発課題「テラヘルツ波リモートセンシング衛星による⽉地下浅部の資源探査」技術...,False,False,False,False,False,False,False,False,False
3,ヒト遺伝学的検査業務委託,False,False,False,False,False,False,False,False,False
4,【入札公告】給食・食器洗浄業務委託,False,False,False,False,False,False,False,False,False
5,令和８年度CE×CNの同時達成に向けた木材再利用の方策等検証事業委託業務【総合評価落札方式】,False,False,False,False,False,False,False,False,False
6,AIテストベッド機能追加,False,False,False,False,False,False,False,True,True
7,空調機器等整備役務,False,False,False,False,False,False,False,False,False
8,【税込入札】国際平和と安全シンポジウム（ハイブリッド形式）の会議運営等業務委託,False,False,False,False,False,False,False,False,False
9,令和８年度自動車事故による重度後遺障害者に係る支援策等の検討に関する調査 【業務委託】,False,False,False,False,False,False,False,False,False


In [38]:
service_notices[tag_columns].sum()

system_development          96
software_applications        6
operations_maintenance     133
cloud                       22
cybersecurity                8
network_infrastructure      80
data_digital                29
artificial_intelligence     24
dtype: int64

In [39]:
service_notices["it_related"].sum()

np.int64(323)

In [41]:
service_notices.loc[
       service_notices["artificial_intelligence"],
       [
           "project_name",
           "organization_name",
           "prefecture_name",
           "cft_issue_date",
       ],
   ]

,project_name,organization_name,prefecture_name,cft_issue_date
6,AIテストベッド機能追加,国立研究開発法人産業技術総合研究所,東京都,2026-07-24T00:00:00+09:00
54,AIチップ設計拠点のネットワークスイッチ増強,国立研究開発法人産業技術総合研究所,東京都,2026-07-08T00:00:00+09:00
154,巨大チップAI半導体LLM推論クラウドの利用および技術相談,国立研究開発法人産業技術総合研究所,東京都,2026-06-09T00:00:00+09:00
164,AIテストベッド運用,国立研究開発法人産業技術総合研究所,東京都,2026-06-05T00:00:00+09:00
237,世田谷区自治体システム標準化に伴う母子保健帳票のAI-OCR導入及び運用保守業務委託に係るプ...,東京都世田谷区,東京都,2026-05-18T00:00:00+09:00
287,「江東区AIデマンド交通実証運行」システム導入業務委託に係る公募型プロポーザルの実施について,東京都江東区,東京都,2026-05-01T00:00:00+09:00
323,生成AI基盤構築に向けたプロトタイプ等の設計・開発を伴う実証業務(PDF/156KB),独立行政法人日本学術振興会,東京都,2026-04-24T00:00:00+09:00
339,アンケート調査「ＡＩの職場導入による働き方への影響等に関する企業調査」実施に係るデータ作成等...,独立行政法人労働政策研究・研修機構,東京都,2026-04-20T00:00:00+09:00
561,AIチップ設計拠点のクラウドサーバ運用保守,国立研究開発法人産業技術総合研究所,東京都,2026-03-03T00:00:00+09:00
562,墨田区ＡＩ相談支援システム導入業務委託に関するプロポーザルを実施します,東京都墨田区,東京都,2026-03-02T00:00:00+09:00


In [43]:
for subgroup in tag_columns:
       matches = service_notices[
           service_notices[subgroup]
       ]

       print(subgroup, len(matches))

       display(
           matches[
               [
                   "project_name",
                   "organization_name",
                   "prefecture_name",
               ]
           ].sample(
               min(20, len(matches)),
               random_state=42,
           )
       )

system_development 96


,project_name,organization_name,prefecture_name
4032,【入札公告】マイナンバー系画面転送システム構築及び保守委託業務,福岡県広川町,福岡県
3706,令和７年度 中部技術排水ポンプ車状態監視システム通信設備更新,国土交通省中部地方整備局,愛知県
2786,研修管理システム導入及び運用管理業務委託契約 一式,独立行政法人国立病院機構久里浜医療センター,神奈川県
5121,バスロケーションシステム等導入委託業務（PDF:172KB）,北海道深川市,北海道
993,産総研情報システム環境に係る導入支援業務,国立研究開発法人産業技術総合研究所,東京都
3863,安城市土地開発公社財務会計システム更新業務委託,愛知県安城市,愛知県
2272,防衛大学校共同利用電子計算機システムのEDR導入役務,防衛省,神奈川県
1374,大阪市統合型校務支援システム等にかかる開発及び運用・保守業務委託,大阪府大阪市,大阪府
1,児童館運営支援システム導入及びシステム運用保守業務委託契約プロポーザルの実施について,東京都世田谷区,東京都
295,【資料提供招請】独立行政法人日本学術振興会業務基盤システム更新・保守業務 一式(PDF/19...,独立行政法人日本学術振興会,東京都


software_applications 6


,project_name,organization_name,prefecture_name
62,令和7年度（補正予算）山地災害調査アプリケーション改修等業務(PDF : 130KB),林野庁,東京都
123,高周波回路シミュレーター用ソフトウェアおよびデバイスモデリング用ソフトウェア保守,国立研究開発法人産業技術総合研究所,東京都
4837,令和６年度福岡国税局におけるアプリケーション開発等支援に係る委託業務,財務省,福岡県
124,高周波回路シミュレーター用ソフトウェア保守,国立研究開発法人産業技術総合研究所,東京都
1975,箕面市保育所入所選考等支援ソフトウェア導入等業務委託に係る総合評価入札方式による一般競争入札...,大阪府箕面市,大阪府
731,令和8年度山地災害調査アプリケーション運用・保守業務(PDF : 257KB),林野庁,東京都


operations_maintenance 133


...

,project_name,organization_name,prefecture_name
2234,第5学生舎等の構内ネットワーク構築役務,防衛省,神奈川県
49,2群2-11A棟サーバ付帯設備の保守点検,国立研究開発法人産業技術総合研究所,東京都
1501,箕面市教育情報ネットワーク（学習系ネットワーク）アセスメント業務委託にかかる一般競争入札の実...,大阪府箕面市,大阪府
2241,防衛大学校第５学生舎へのネットワーク機器設定役務,防衛省,神奈川県
954,時間確定性を高めるネットワーク制御技術の性能評価にかかる技術相談の対応,国立研究開発法人産業技術総合研究所,東京都
2129,労災疾病等医学研 究・開発、普及ネットワークシステム等管理及びコンサルティングに関する業務委...,独立行政法人労働者健康安全機構,神奈川県
561,AIチップ設計拠点のクラウドサーバ運用保守,国立研究開発法人産業技術総合研究所,東京都
5034,令和8年度データセンター集積推進事業（海外データセンター誘致）委託業務に係る総合評価一般競争...,北海道,北海道
262,Ｃ３棟フロア移設に伴う公刊情報収集装置のネットワーク構築・Ｌ２スイッチ設定等役務,防衛省,東京都
744,エッジ側データ処理サービスの技術相談・開発支援、およびエッジデバイス群、エッジサーバ群の試験...,国立研究開発法人産業技術総合研究所,東京都


data_digital 29


,project_name,organization_name,prefecture_name
5374,石狩市下水道管路台帳デジタル化業務委託公募型プロポーザルの実施について,北海道石狩市,北海道
1719,令和7年度都市・まちDX推進に向けた建設生産プロセスDX推進支援業務委託,大阪府大阪市,大阪府
1325,令和8年度大阪市DX戦略実行支援業務委託,大阪府大阪市,大阪府
5002,一般競争入札の実施（令和８年度(2026年度)農業DX人材育成プログラム動画制作・配信委託業務）,北海道,北海道
1246,令和8年度大阪市DX戦略推進施策伴走支援業務委託,大阪府大阪市,大阪府
1249,大阪市データ連携ツール導入業務委託,大阪府大阪市,大阪府
3922,令和７年度 狩野川河川現況台帳デジタル化作業,国土交通省中部地方整備局,愛知県
28,【公募型プロポーザル】ICT化・DX化による介護事業所の業務負担軽減支援業務委託,東京都板橋区,東京都
5218,【公告】令和8年度DX関連産業振興事業委託業務に係る総合評価一般競争入札実施のお知らせ,北海道,北海道
1329,大阪市自治体窓口DXSaaS環境構築及び運用保守業務委託,大阪府大阪市,大阪府


artificial_intelligence 24


,project_name,organization_name,prefecture_name
561,AIチップ設計拠点のクラウドサーバ運用保守,国立研究開発法人産業技術総合研究所,東京都
1272,AI英語学習業務委託（中学校）にかかる総合評価落札方式による一般競争入札の実施について,大阪府箕面市,大阪府
6,AIテストベッド機能追加,国立研究開発法人産業技術総合研究所,東京都
1977,箕面市生成AI利用環境構築及び運用支援業務委託に係る総合評価入札方式による一般競争入札の実施...,大阪府箕面市,大阪府
756,医療現場における AI 技術等を活用した業務効率化の効果検証等にかかる業務委託（PDF：13...,独立行政法人国立病院機構,東京都
562,墨田区ＡＩ相談支援システム導入業務委託に関するプロポーザルを実施します,東京都墨田区,東京都
814,東京出入国在留管理局におけるＡＩ翻訳及び電話通訳等業務委託契約,法務省,東京都
54,AIチップ設計拠点のネットワークスイッチ増強,国立研究開発法人産業技術総合研究所,東京都
3053,高齢者のAIリテラシーに関する全国アンケート調査業務委託,国立研究開発法人国立長寿医療研究センター,愛知県
287,「江東区AIデマンド交通実証運行」システム導入業務委託に係る公募型プロポーザルの実施について,東京都江東区,東京都


In [45]:
core_it_columns = [
       "system_development",
       "software_applications",
       "operations_maintenance",
       "cloud",
       "cybersecurity",
       "network_infrastructure",
   ]

service_notices["core_it_delivery"] = (
       service_notices[core_it_columns]
       .any(axis=1)
   )

service_notices["digital_or_ai_adjacent"] = (
       (
           service_notices["data_digital"]
           | service_notices["artificial_intelligence"]
       )
       & ~service_notices["core_it_delivery"]
   )

In [51]:
service_notices[
       [
           "project_name",
           "core_it_delivery",
           "digital_or_ai_adjacent",
       ]
   ].sample(50, random_state=42)

,project_name,core_it_delivery,digital_or_ai_adjacent
1782,大阪大学病児・病後児保育室運営委託業務 一式,False,False
3917,入札公告（名古屋法務合同庁舎及び法務総合研究所名古屋支所清掃業務委託契約）,False,False
221,2026-2028年度JICA海外協力隊選考時健康判定にかかる委託業務（25a00580）（...,False,False
2135,南関東防衛局管内(８)駐留軍等労働者雇用前健康診断業務委託（座間地区）,False,False
5224,上富良野演習場内試験施工施設詳細設計委託業務,False,False
1168,令和８年度第一次収穫調査業務委託（４号物件：大箕山国有林外）,False,False
879,麻布地区地域情報紙「ザ・AZABU」編集業務委託事業候補者をプロポーザル方式により募集します,False,False
156,江戸川区小岩アーバンプラザ舞台操作業務委託事業者選定プロポーザルの実施,False,False
1657,令和7年度定期健康診断業務委託,False,False
323,生成AI基盤構築に向けたプロトタイプ等の設計・開発を伴う実証業務(PDF/156KB),False,True


In [50]:
service_notices.shape

(6000, 19)

In [53]:
unmatched = service_notices[
       ~service_notices["it_related"]
   ]

unmatched[
       [
           "project_name",
           "organization_name",
           "prefecture_name",
       ]
   ].sample(100, random_state=42)

,project_name,organization_name,prefecture_name
1248,令和8年度大阪刑務所職員一般及び特別定期健康診断等業務委託契約,法務省,大阪府
1042,令和8年度水道配水管設計業務委託（No．3）,大阪府豊中市,大阪府
1948,令和6年度下水道管内詳細調査業務委託（No．1）,大阪府豊中市,大阪府
3118,令和８年度 三重河川国道事務所低濃度ＰＣＢ廃棄物処理作業,国土交通省中部地方整備局,愛知県
5276,配水管整備事業 鉄西中央幹線 測量及び実施設計業務委託【5月19日】,北海道釧路市,北海道
...,...,...,...
3078,令和８年度 ＢＩＭＣＩＭ研修運営補助業務,国土交通省中部地方整備局,愛知県
218,浅川左岸第一処理分区ほか(R8-1)管渠更生実施設計業務委託,東京都日野市,東京都
1224,令和８年度大阪教育大学附属学校園ＩＣＴ支援業務委託 一式,国立大学法人大阪教育大学,大阪府
2122,電話交換業務委託,独立行政法人労働者健康安全機構横浜労災病院,神奈川県


In [55]:
service_notices.columns.tolist()

['key',
 'project_name',
 'organization_name',
 'prefecture_name',
 'category',
 'cft_issue_date',
 'external_document_uri',
 'data_json',
 'system_development',
 'software_applications',
 'operations_maintenance',
 'cloud',
 'cybersecurity',
 'network_infrastructure',
 'data_digital',
 'it_related',
 'artificial_intelligence',
 'core_it_delivery',
 'digital_or_ai_adjacent']

In [58]:
it_notice_count = int(
       service_notices["it_related"].sum()
   )

it_share_pct = (
       service_notices["it_related"].mean() * 100
   )

print("IT-related notices:", it_notice_count)
print("Service notices:", len(service_notices))
print("IT share:", round(it_share_pct, 2), "%")

IT-related notices: 323
Service notices: 6000
IT share: 5.38 %


In [60]:
subgroup_summary = (
       service_notices[tag_columns]
       .sum()
       .sort_values(ascending=False)
   )

subgroup_summary

operations_maintenance     133
system_development          96
network_infrastructure      80
data_digital                29
artificial_intelligence     24
cloud                       22
cybersecurity                8
software_applications        6
dtype: int64

In [61]:
current_it_notices = service_notices[
       service_notices["it_related"]
   ]

current_top_organizations = (
       current_it_notices["organization_name"]
       .value_counts()
       .head(20)
   )

current_top_organizations

organization_name
大阪府大阪市                       32
国立研究開発法人産業技術総合研究所            24
福岡県                          19
北海道                          19
防衛省                          18
大阪府箕面市                       17
国土交通省中部地方整備局                 15
環境省                          11
国立研究開発法人新エネルギー・産業技術総合開発機構    11
国立研究開発法人国立長寿医療研究センター         11
国立大学法人東京工業大学                  8
独立行政法人労働者健康安全機構               8
東京都世田谷区                       6
林野庁                           6
独立行政法人日本学術振興会                 4
国立研究開発法人国立がん研究センター            4
独立行政法人国際協力機構                  4
防衛省防衛大学校                      4
国土交通省                         4
国家公安委員会（警察庁）福岡県警察             4
Name: count, dtype: int64